In [5]:
import sys
import os
import MySQLdb

MySQLdb._mysql
# Get the current working directory of the notebook
notebook_dir = os.getcwd()

# Go up one level to the 'root_directory'
root_dir = os.path.dirname(notebook_dir)

# Add the root directory to sys.path
sys.path.append(root_dir)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "djangowork.settings.dev")

os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")

# Setup Django
import django

django.setup()

# Now you can import your model
from store.models import Product

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
def get_similar_products(product_pk: int):
    """
    Returns a list of top 6 similar Product objects based on text similarity.

    Args:
        product_pk (int): The pk of the product to find recommendations for.

    Returns:
        QuerySet: A queryset of similar Product instances.
    """

    # Fetch product data from the database
    products = Product.objects.all().values('pk', 'title', 'description')

    # Convert product data to DataFrame
    df = pd.DataFrame(list(products))

    # Combine title and description into one string per product
    df['features'] = df['title'] + " " + df['description']

    df['features'] = df['features'].fillna("")

    # Convert text to TF-IDF vectors
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(df['features'])

    # Check if the given product_pk exists
    if product_pk not in df['pk'].values:
        return []

    # Find index of the product
    idx = df[df['pk'] == product_pk].index[0]

    # Calculate cosine similarity of this product with all products
    similarity_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix)

    # Getting top 5 similar indices
    similar_indices = similarity_score.argsort()[0][-6:-1]

    #Getting top 6 similar Products
    similar_titles = df.iloc[similar_indices]['title'].tolist()

    return similar_titles

In [24]:
print(get_similar_products(6))
print(Product.objects.filter(pk = 6).values("title", "pk"))

['Mustard - Individual Pkg', 'Beef - Sushi Flat Iron Steak', 'Sunflower Seed Raw', 'Cranberries - Dry', 'Mayonnaise - Individual Pkg']
<QuerySet [{'title': 'Mustard - Individual Pkg', 'pk': 6}]>
